# E. coli growth at three temperatures

This notebook reads the growth curves of *E. coli* at 29, 37 and 45 °C with 1 % glucose
from `../data/raw/`, writes processed tables to `../data/processed/` and figures to
`../results/figures/`, and fetches more workbooks of the dataset from Zenodo.

In [ ]:
# Import the modules: pathlib for paths, numpy for arrays, pandas for tables, matplotlib for plots
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Paths, relative to this notebook in code/
RAW = Path("../data/raw")              # data as it came from its source: never changed
PROCESSED = Path("../data/processed")  # tables computed from raw data
FIGURES = Path("../results/figures")   # figures

In [ ]:
# Read the growth curve at 37 °C and show its first rows
growth_37 = pd.read_csv(RAW / "ecoli-growth-37c-1pct.csv")
growth_37.head()

In [ ]:
# numpy arrays: the culture time in hours and the optical density at 37 °C
hours = growth_37["tcultivo (min)"].to_numpy() / 60   # one division for all readings
od = growth_37["Average"].to_numpy()
print(type(hours), hours.shape)

# The first time the optical density is above 0.4
above = od > 0.4                                      # an array of True and False
print(f"OD above 0.4 after {hours[above][0]:.1f} h")

## EX1: Read, write and plot (15 min, on your own)

You want to see how temperature changes the growth of *E. coli*. Read the three
temperature files, keep a processed copy with the time in hours, and plot the three curves
in one figure.

1. Run the first four code cells (Shift + Enter). You should see the first rows of the
   37 °C table, then `<class 'numpy.ndarray'> (289,)` and `OD above 0.4 after 2.3 h`.
   `hours` is a numpy array: one division turns all 289 readings into hours at once.
2. Copy the cells `# Import the modules` and `# Paths` into the chat and ask for the
   code, for example:

   > Using `RAW` and `PROCESSED` above, write pandas code that reads
   > `ecoli-growth-29c-1pct.csv`, `ecoli-growth-37c-1pct.csv` and
   > `ecoli-growth-45c-1pct.csv`, adds a column `time_h` with `tcultivo (min)` in hours,
   > keeps the tables in a dictionary `growth` with keys such as `"37c"`, and writes each
   > table to `PROCESSED` as `ecoli-growth-<temperature>-1pct-hours.csv`. Create
   > `PROCESSED` if it is missing.

3. Paste the code into the cell `# your task: read …` and run it. In the file browser,
   `data/processed/` should hold three files such as `ecoli-growth-37c-1pct-hours.csv`.
   Open one: its last column is `time_h`, ending near 24.1.
4. Ask for the figure, for example:

   > Plot the column `Average` against `time_h` for every table in `growth`, in one
   > figure, with axis labels that name the units, a legend with the temperatures and a
   > title. Save it to `FIGURES` as `growth-curves-1pct.png`. Create `FIGURES` if it is
   > missing.

   Run it in the cell `# your task: plot …`. You should see three curves: 37 °C rises
   first, 29 °C last, and 45 °C levels off lowest. `results/figures/` should hold the PNG
   file.
5. Save the notebook (Ctrl + S) and open the Git panel. It should list
   `growth_curves.ipynb` as changed, and the three processed files and the figure as
   untracked. Stage them all, write a short message, for example
   `plot growth at three temperatures`, and commit.

In [ ]:
# your task: read the three temperatures, add the time in hours, write them to PROCESSED
PROCESSED.mkdir(parents=True, exist_ok=True)

growth = {}
for temperature in ["29c", "37c", "45c"]:
    table = pd.read_csv(RAW / f"ecoli-growth-{temperature}-1pct.csv")
    table["time_h"] = table["tcultivo (min)"] / 60
    table.to_csv(PROCESSED / f"ecoli-growth-{temperature}-1pct-hours.csv", index=False)
    growth[temperature] = table
    print(temperature, len(table), "readings, time_h up to", round(table["time_h"].max(), 2))

In [ ]:
# your task: plot the three growth curves in one figure and save it to FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots()
for temperature, table in growth.items():
    ax.plot(table["time_h"].to_numpy(), table["Average"].to_numpy(),
            label=temperature.replace("c", " °C"))
ax.set_xlabel("Culture time (h)")
ax.set_ylabel("Optical density (mean of 8 wells)")
ax.set_title("E. coli with 1 % glucose at three temperatures")
ax.legend(title="Temperature")
fig.savefig(FIGURES / "growth-curves-1pct.png", dpi=150)

## EX2: Fetch data from a repository (20 min, on your own)

Your figure shows 1 % glucose only, and the dataset's repository also has a workbook for
2 % glucose. You fetch it with code, so the step can be rerun and checked, and you make
sure that the file arrived intact and your raw data stayed as it was.

1. Run the cell `# Ask Zenodo for the record`. You should see the record's title and its
   five files, each with its size and an `md5:` checksum. `record` holds the whole answer
   as dictionaries and lists.
2. Ask the LLM for the download, for example:

   > `record["files"]` is a list of dictionaries with the keys `key` (the file name),
   > `checksum` (such as `"md5:4b23…"`) and `links`, where `links["self"]` is the
   > download URL. Write Python with `urllib.request` and `hashlib` that downloads
   > `Repo2pct.xlsx`, computes its MD5 and saves it to `RAW` only if the MD5 matches the
   > checksum. Print both checksums.

   Run it in the cell `# your task: download …`. You should see the same checksum twice,
   and `Repo2pct.xlsx` in `data/raw/`.
3. Ask for the conversion to an open format, for example:

   > Write pandas code that reads every sheet of `RAW / "Repo2pct.xlsx"` and writes each
   > one to `PROCESSED` as CSV without the index, named like `ecoli-growth-29c-2pct.csv`
   > for a sheet named like `29_2%`.

   Run it in the cell `# your task: convert …`. `data/processed/` should hold nine new
   files, from `ecoli-growth-29c-2pct.csv` to `ecoli-growth-45c-2pct.csv`. The warning
   `Workbook contains no default style` is harmless.
4. Run the cell `# Check in Git`. `git status` should list `Repo2pct.xlsx` as new (`??`)
   and no changed file (`M`) in `data/raw/`. `git log` shows the history of `data/raw/`:
   the one commit of the setup, which added the raw data.
5. Save the notebook and open the Git panel. It should list `growth_curves.ipynb`,
   `Repo2pct.xlsx` and the nine CSV files. Stage them all, write a short message, for
   example `fetch the 2 % glucose workbook from Zenodo`, and commit.

In [ ]:
# Ask Zenodo for the record: its API answers in JSON, which json.load turns into dictionaries and lists
import json
import urllib.request

RECORD_URL = "https://zenodo.org/api/records/19416053"
with urllib.request.urlopen(RECORD_URL, timeout=60) as response:
    record = json.load(response)

print(record["metadata"]["title"])
for file in record["files"]:
    print(file["key"], file["size"], "bytes", file["checksum"])

In [ ]:
# your task: download Repo2pct.xlsx, check its MD5 and save it to RAW
import hashlib

wanted = "Repo2pct.xlsx"
entry = next(file for file in record["files"] if file["key"] == wanted)
expected = entry["checksum"].removeprefix("md5:")

with urllib.request.urlopen(entry["links"]["self"], timeout=60) as response:
    content = response.read()
actual = hashlib.md5(content).hexdigest()
print("expected", expected)
print("actual  ", actual)

if actual == expected:
    (RAW / wanted).write_bytes(content)
else:
    print("The checksums differ, so the file was not saved.")

In [ ]:
# your task: convert every sheet of Repo2pct.xlsx to CSV in PROCESSED
sheets = pd.read_excel(RAW / "Repo2pct.xlsx", sheet_name=None)   # all sheets, by sheet name
for sheet_name, table in sheets.items():
    temperature = sheet_name.split("_")[0]                         # "29_2%" -> "29"
    target = PROCESSED / f"ecoli-growth-{temperature}c-2pct.csv"
    table.to_csv(target, index=False)
    print(sheet_name, "->", target.name)

In [ ]:
# Check in Git: what is new (??) or changed (M) in data/raw, then the history of data/raw
# (core.quotePath=false shows a name such as Repo0·1pct.xlsx as it is)
!git -c core.quotePath=false status --short ../data/raw
!git log --oneline -- ../data/raw

## EX3: Fetch in a loop (30 min, homework)

A later analysis will compare all four glucose concentrations. Fetch their workbooks in
one loop that checks every file, so that you can rerun it at any time, and record the
files in the experiment's metadata.

1. After a kernel restart, run all cells above this one first (Run → Run All Above
   Selected Cell). Then ask the LLM to turn your download code into a loop over each file
   in `record["files"]` whose name starts with `Repo`: a file already in `RAW` gets its
   MD5 checked, a missing file is downloaded, checked and saved. Each file gets one
   printed line: its name, `present` or `downloaded`, and whether the checksum matches.
2. Run it in the cell `# your task: fetch in a loop`. You should see four lines:
   `Repo1pct.xlsx` and `Repo2pct.xlsx` present, `Repo0·1pct.xlsx` and `Repo0·5pct.xlsx`
   downloaded, all four checksums matching. Run it once more: all four are present and
   nothing is downloaded. The repository limits how many requests it answers per minute,
   so do not run the loop over and over.
3. Run the cell `# Check in Git` again. You should see the two new workbooks (`??`) and
   no changed file.
4. In `metadata.yml`, list the two new workbooks and `Repo2pct.xlsx` under `inputs`, the
   way `Repo1pct.xlsx` is listed, with the checksums from the record. List the processed
   files and the figure under `outputs`, and fill in `date_started`, `author` and
   `status`.
5. In `README.md`, replace the placeholders under "Data generated or processed" and answer
   the questions under "Observations and interpretation". Add an entry for today at the
   top of `journal.md` (in `work/rrw-workshop/`): what you fetched and how you checked it.
6. Save the notebook and open the Git panel. It should list `growth_curves.ipynb`, the two
   workbooks, `metadata.yml`, `README.md` and `journal.md`. Write a short message, for
   example `fetch all glucose workbooks in a loop`, and commit.

In [ ]:
# your task: fetch in a loop
def md5_of(content):
    return hashlib.md5(content).hexdigest()

for entry in record["files"]:
    name = entry["key"]
    if not name.startswith("Repo"):
        continue
    expected = entry["checksum"].removeprefix("md5:")
    target = RAW / name

    if target.exists():
        state = "present"
        actual = md5_of(target.read_bytes())
    else:
        state = "downloaded"
        with urllib.request.urlopen(entry["links"]["self"], timeout=60) as response:
            content = response.read()
        actual = md5_of(content)
        if actual == expected:
            target.write_bytes(content)

    result = "checksum ok" if actual == expected else f"checksum differs: {actual}, not {expected}"
    print(name, state, result)